# Repeated cross-validation and classifier experiments

Compare supervised logistic regression, DBSCAN, and GMM, then assess stopping tolerance, embedding normalization, and DBSCAN–GMM probability blending. Primary metric: sample-wise F1 with the existing threshold and nonempty fallback.

## Setup and predeclared comparisons

Use the existing 157 labeled training trials and keep the 78-trial validation set separate. The full grid and fold seeds are fixed before scores are computed. Requires iterative-stratification==0.1.9 in addition to the existing environment.

In [1]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import platform
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.base import clone
from sklearn.cluster import DBSCAN
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import Normalizer, StandardScaler
from threadpoolctl import threadpool_limits

REASONS = ['Safety', 'Funding', 'Efficacy', 'Recruitment', 'Other']
FEATURES = [f'emb_{i}' for i in range(150)]
SPLIT_SEEDS = [42, 43, 44, 45, 46]
N_FOLDS = 3
TOLERANCES = [1e-3, 3e-4, 1e-4, 3e-5, 1e-5]
NORMALIZED_CS = [1., 10., 100., 1000., 10000., 100000.]
GMM_BLEND_WEIGHTS = [0.25, 0.5, 0.75]
DEFAULT_C = 1000.
DEFAULT_TOL = 1e-4
LABEL_RULE = dict(minimum_support=3., prior_strength=2., positive_threshold=0.6,
                  negative_threshold=0.1, maximum_generated_weight=0.25,
                  maximum_class_fraction=0.5)
GMM_RULE = dict(n_components=10, covariance_type='tied', n_init=3, max_iter=500,
                tol=1e-3, reg_covar=1e-3, random_state=42, init_params='kmeans')

def token(value):
    return f'{value:g}'.replace('.', 'p').replace('-', 'm')

CONFIGS = [
    dict(name='supervised', stage='baseline', dataset='supervised', C=DEFAULT_C, tol=DEFAULT_TOL, normalize=False),
    dict(name='dbscan', stage='baseline', dataset='dbscan', C=DEFAULT_C, tol=DEFAULT_TOL, normalize=False),
    dict(name='gmm', stage='baseline', dataset='gmm', C=DEFAULT_C, tol=DEFAULT_TOL, normalize=False),
]
CONFIGS += [dict(name=f'tolerance_{token(t)}', stage='tolerance', dataset='dbscan', C=DEFAULT_C,
                 tol=t, normalize=False) for t in TOLERANCES if t != DEFAULT_TOL]
CONFIGS += [dict(name=f'normalized_C{token(c)}', stage='normalization', dataset='dbscan', C=c,
                 tol=DEFAULT_TOL, normalize=True) for c in NORMALIZED_CS]
CONFIGS += [dict(name=f'blend_gmm_{token(w)}', stage='blend', gmm_weight=w) for w in GMM_BLEND_WEIGHTS]
assert len(CONFIGS) == 16 and len({c['name'] for c in CONFIGS}) == 16

cwd = Path.cwd()
DATA = next(p for p in [cwd, cwd/'part1', cwd/'AML-Midterm'/'part1']
            if (p/'part1_train.csv').is_file() or (p/'data/part1_train.csv').is_file()).resolve()
INPUT = DATA if (DATA/'part1_train.csv').is_file() else DATA/'data'
OUT = DATA/'experiments/classifier_cross_validation'
for folder in ['splits', 'contexts', 'models', 'predictions', 'datasets']:
    (OUT/folder).mkdir(parents=True, exist_ok=True)
train = pd.read_csv(INPUT/'part1_train.csv')
valid = pd.read_csv(INPUT/'part1_val.csv')
X = train[FEATURES].to_numpy()
Y = train[REASONS].to_numpy(dtype=int)
known = np.isin(Y, [0, 1]).all(axis=1)
known_rows = np.flatnonzero(known)
unlabeled_rows = np.flatnonzero(~known)
XV = valid[FEATURES].to_numpy()
YV = valid[REASONS].to_numpy(dtype=int)
assert len(train) == 1171 and len(valid) == 78 and known.sum() == 157
assert np.isfinite(X).all() and np.isfinite(XV).all()
assert np.isin(Y[known], [0, 1]).all() and (Y[~known] == -100).all()
assert train.ID.is_unique and valid.ID.is_unique and set(train.ID).isdisjoint(valid.ID)
SOURCE = DATA/'experiments/additional_methods'
DB_MODEL = SOURCE/'classifier_models/dbscan_m5_q25__logistic_C1000.joblib'
GMM_MODEL = DATA/'experiments/method_comparison/classifier_models/gmm_tied_k10__logistic_C1000.joblib'
SUPERVISED_MODEL = DATA/'experiments/kmeans/models/baseline_C1000.joblib'
protected = [INPUT/'part1_train.csv', INPUT/'part1_val.csv', DB_MODEL, GMM_MODEL, SUPERVISED_MODEL,
             SOURCE/'datasets/dbscan_m5_q25.csv',
             DATA/'experiments/method_comparison/datasets/gmm_tied_k10.csv',
             SOURCE/'submissions/part1_dbscan_m5_q25_logistic_C1000_submission.csv',
             SOURCE/'submission_manifest.json']
PROTECTED_HASHES = {str(p.relative_to(DATA)): hashlib.sha256(p.read_bytes()).hexdigest() for p in protected}
train[['ID']+FEATURES+REASONS].to_csv(OUT/'datasets/original_training.csv', index=False)
valid[['ID']+FEATURES+REASONS].to_csv(OUT/'datasets/external_validation.csv', index=False)
print('Predeclared: 5 repeats x 3 folds; 3 baseline methods; 4 new tolerances; 6 normalized C values; 3 blends.')
print('All fitting excludes the 78 external validation trials. No test data, final refit, or submission.')



Predeclared: 5 repeats x 3 folds; 3 baseline methods; 4 new tolerances; 6 normalized C values; 3 blends.
All fitting excludes the 78 external validation trials. No test data, final refit, or submission.


## Fitting and scoring helpers

Every fold excludes its held-out trials from all fitting. Clustering uses the remaining verified trials plus the 1,014 originally unlabeled training trials. Generated labels stay fixed within a fold for classifier changes.

In [2]:
def label_predictions(probability):
    probability = np.asarray(probability)
    assert probability.ndim == 2 and probability.shape[1] == 5
    assert np.isfinite(probability).all() and ((probability >= 0) & (probability <= 1)).all()
    prediction = (probability >= 0.5).astype(int)
    empty = prediction.sum(axis=1) == 0
    rows = np.flatnonzero(empty)
    prediction[rows, probability[rows].argmax(axis=1)] = 1
    return prediction, empty


def score(truth, probability):
    prediction, empty = label_predictions(probability)
    return dict(sample_f1=float(f1_score(truth, prediction, average='samples', zero_division=0)),
                macro_f1=float(f1_score(truth, prediction, average='macro', zero_division=0)),
                fallback_trials=int(empty.sum()),
                **dict(zip([f'f1_{r}' for r in REASONS],
                           f1_score(truth, prediction, average=None, zero_division=0).tolist())))


def predict_model(packet, raw):
    scaled = packet['scaler'].transform(np.asarray(raw))
    if packet.get('normalizer') is not None:
        scaled = packet['normalizer'].transform(scaled)
    return np.column_stack([e.predict_proba(scaled)[:, list(e.classes_).index(1)]
                            for e in packet['estimators']])


def pseudo_targets(labels, is_known, probability, support):
    targets = np.full(labels.shape, -100, dtype=int)
    weights = np.zeros(labels.shape)
    targets[is_known] = labels[is_known]
    weights[is_known] = 1.
    eligible = (~is_known) & (support >= LABEL_RULE['minimum_support'])
    for j in range(5):
        positive = eligible & (probability[:, j] >= LABEL_RULE['positive_threshold'])
        negative = eligible & (probability[:, j] <= LABEL_RULE['negative_threshold'])
        np_, nn_ = int(positive.sum()), int(negative.sum())
        kp = int(labels[is_known, j].sum())
        kn = int(is_known.sum() - kp)
        assert kp > 0 and kn > 0
        if not np_ or not nn_:
            continue
        fraction = min(LABEL_RULE['maximum_class_fraction'],
                       LABEL_RULE['maximum_generated_weight']*np_/kp,
                       LABEL_RULE['maximum_generated_weight']*nn_/kn)
        targets[positive, j] = 1
        targets[negative, j] = 0
        weights[positive, j] = fraction*kp/np_
        weights[negative, j] = fraction*kn/nn_
        assert weights[~is_known, j].max() <= 0.25 + 1e-12
        np.testing.assert_allclose(weights[positive, j].sum()/kp,
                                   weights[negative, j].sum()/kn, atol=1e-12)
    assert np.array_equal(weights > 0, targets != -100)
    np.testing.assert_array_equal(targets[is_known], labels[is_known])
    return targets, weights


def cluster_probabilities(responsibility, labels, is_known):
    support_by_component = responsibility[is_known].sum(axis=0)
    positives = responsibility[is_known].T @ labels[is_known]
    prior = labels[is_known].mean(axis=0)
    rates = (positives + LABEL_RULE['prior_strength']*prior) / (
        support_by_component[:, None] + LABEL_RULE['prior_strength'])
    return responsibility @ rates, responsibility @ support_by_component


CONTEXT_AUDIT = []
LABEL_AUDIT = []
FIT_AUDIT = []

def build_context(name, fitting_known_rows, held_rows, repeat=None, fold=None):
    start = time.monotonic()
    # No held-out trial appears even as an unlabeled row in the fitting pool.
    pool = np.sort(np.concatenate([fitting_known_rows, unlabeled_rows]))
    assert len(np.intersect1d(pool, held_rows)) == 0
    assert set(pool) == set(fitting_known_rows) | set(unlabeled_rows)
    local_known = known[pool]
    labels = Y[pool].copy()
    assert local_known.sum() == len(fitting_known_rows)
    scaler = StandardScaler().fit(X[fitting_known_rows])
    cluster_scaler = StandardScaler().fit(X[pool])
    scaled = scaler.transform(X[pool])
    clustered = cluster_scaler.transform(X[pool])
    normalizer = Normalizer(norm='l2').fit(scaled[local_known])
    normalized = normalizer.transform(scaled)
    np.testing.assert_allclose(np.linalg.norm(normalized, axis=1), 1., atol=1e-12)
    target0 = np.full(labels.shape, -100, dtype=int)
    weight0 = np.zeros(labels.shape)
    target0[local_known] = labels[local_known]
    weight0[local_known] = 1.
    datasets = {'supervised': (target0, weight0)}
    with threadpool_limits(limits=1):
        distances = NearestNeighbors(n_neighbors=5, algorithm='brute', n_jobs=1).fit(clustered).kneighbors(clustered)[0][:, -1]
        eps = max(float(np.quantile(distances, 0.25)), 1e-6)
        db = DBSCAN(eps=eps, min_samples=5, metric='euclidean', algorithm='brute', n_jobs=1).fit(clustered)
        unique = np.unique(db.labels_[db.labels_ >= 0])
        responsibility = np.column_stack([db.labels_ == c for c in unique]).astype(float) if len(unique) else np.empty((len(pool), 0))
        probability, support = cluster_probabilities(responsibility, labels, local_known)
        datasets['dbscan'] = pseudo_targets(labels, local_known, probability, support)
        assert (datasets['dbscan'][1][(db.labels_ < 0) & ~local_known] == 0).all()
        gmm = GaussianMixture(**GMM_RULE)
        with warnings.catch_warnings():
            warnings.simplefilter('error', ConvergenceWarning)
            gmm.fit(clustered)
        assert gmm.converged_
        responsibility_gmm = gmm.predict_proba(clustered)
        np.testing.assert_allclose(responsibility_gmm.sum(axis=1), 1., atol=1e-10)
        probability_gmm, support_gmm = cluster_probabilities(responsibility_gmm, labels, local_known)
        datasets['gmm'] = pseudo_targets(labels, local_known, probability_gmm, support_gmm)
    directory = OUT/'contexts'/name
    directory.mkdir(exist_ok=True)
    roles = train[['ID']].copy()
    roles['role'] = 'held_out'
    roles.loc[unlabeled_rows, 'role'] = 'unlabeled_fit'
    roles.loc[fitting_known_rows, 'role'] = 'verified_fit'
    roles.to_csv(directory/'row_roles.csv', index=False)
    for dataset, (targets, weights) in datasets.items():
        frame = train.iloc[pool][['ID']].copy()
        frame['is_verified_fit'] = local_known
        for j, reason in enumerate(REASONS):
            frame[reason] = targets[:, j]
            frame[f'weight_{reason}'] = weights[:, j]
            frame[f'normalized_weight_{reason}'] = weights[:, j]*local_known.sum()/weights[:, j].sum()
            pos = (~local_known) & (targets[:, j] == 1)
            neg = (~local_known) & (targets[:, j] == 0)
            LABEL_AUDIT.append(dict(context=name, repeat=repeat, fold=fold, dataset=dataset, reason=reason,
                                    known_positive=int(labels[local_known, j].sum()),
                                    known_negative=int((labels[local_known, j] == 0).sum()),
                                    pseudo_positive=int(pos.sum()), pseudo_negative=int(neg.sum()),
                                    pseudo_positive_weight=float(weights[pos, j].sum()),
                                    pseudo_negative_weight=float(weights[neg, j].sum())))
        frame.to_csv(directory/f'{dataset}_targets_weights.csv', index=False)
    # Persist cluster memberships/responsibilities and support for audit/replay.
    cluster_frame = train.iloc[pool][['ID']].copy()
    cluster_frame['dbscan_cluster'] = db.labels_
    cluster_frame['dbscan_support'] = support
    cluster_frame['gmm_support'] = support_gmm
    for j, reason in enumerate(REASONS):
        cluster_frame[f'dbscan_probability_{reason}'] = probability[:, j]
        cluster_frame[f'gmm_probability_{reason}'] = probability_gmm[:, j]
    for k in range(10):
        cluster_frame[f'gmm_responsibility_{k}'] = responsibility_gmm[:, k]
    cluster_frame.to_csv(directory/'clustering_audit.csv', index=False)
    joblib.dump(dict(scaler=scaler, cluster_scaler=cluster_scaler, normalizer=normalizer,
                     dbscan=db, gmm=gmm, fitting_pool_rows=pool, fitting_known_rows=fitting_known_rows,
                     held_rows=held_rows, label_rule=LABEL_RULE), directory/'preprocessing_clustering.joblib', compress=3)
    audit = dict(context=name, repeat=repeat, fold=fold, verified_fit_rows=len(fitting_known_rows),
                 unlabeled_fit_rows=len(unlabeled_rows), held_rows=len(held_rows),
                 scaler_fit_rows=int(scaler.n_samples_seen_), cluster_fit_rows=int(cluster_scaler.n_samples_seen_),
                 heldout_fit_overlap=0, dbscan_eps=eps, dbscan_clusters=len(unique),
                 dbscan_noise=int((db.labels_ < 0).sum()), gmm_iterations=int(gmm.n_iter_),
                 gmm_converged=bool(gmm.converged_), elapsed_seconds=time.monotonic()-start)
    CONTEXT_AUDIT.append(audit)
    return dict(name=name, repeat=repeat, fold=fold, pool=pool, known=local_known, datasets=datasets,
                scaler=scaler, normalizer=normalizer, scaled=scaled, normalized=normalized,
                cache={}, models={}, probabilities={}, held_rows=held_rows,
                fitting_known_rows=fitting_known_rows)


def fit_config(context, config):
    name = config['name']
    if name in context['probabilities']:
        return context['probabilities'][name]
    if config['stage'] == 'blend':
        weight = config['gmm_weight']
        probability = (1.-weight)*context['probabilities']['dbscan'] + weight*context['probabilities']['gmm']
        packet = dict(kind='probability_blend', gmm_weight=weight,
                      components=['dbscan', 'gmm'], component_files=[f'dbscan.joblib', f'gmm.joblib'])
    else:
        targets, weights = context['datasets'][config['dataset']]
        normalized_weights = weights*context['known'].sum()/weights.sum(axis=0)
        np.testing.assert_allclose(normalized_weights.sum(axis=0), context['known'].sum(), atol=1e-10)
        design = context['normalized'] if config['normalize'] else context['scaled']
        estimators = []
        for j, reason in enumerate(REASONS):
            included = weights[:, j] > 0
            key = (j, config['C'], config['tol'], config['normalize'],
                   hashlib.sha256(targets[:, j].tobytes()+normalized_weights[:, j].tobytes()).hexdigest())
            if key not in context['cache']:
                estimator = LogisticRegression(C=config['C'], tol=config['tol'], solver='lbfgs',
                                               max_iter=5000, random_state=42, class_weight=None)
                with threadpool_limits(limits=1), warnings.catch_warnings():
                    warnings.simplefilter('error', ConvergenceWarning)
                    estimator.fit(design[included], targets[included, j], sample_weight=normalized_weights[included, j])
                context['cache'][key] = estimator
                FIT_AUDIT.append(dict(context=context['name'], first_config=name, reason=reason,
                                      C=config['C'], tol=config['tol'], normalized=config['normalize'],
                                      training_rows=int(included.sum()), known_rows=int(context['known'].sum()),
                                      weight_total=float(normalized_weights[included, j].sum()),
                                      iterations=int(estimator.n_iter_.max()),
                                      coefficient_norm=float(np.linalg.norm(estimator.coef_))))
            estimators.append(context['cache'][key])
        packet = dict(kind='logistic', scaler=context['scaler'],
                      normalizer=context['normalizer'] if config['normalize'] else None,
                      estimators=estimators, config=config, features=FEATURES, reasons=REASONS,
                      normalized_weight_total=int(context['known'].sum()), context=context['name'])
        raw = XV if context['name'] == 'full_training' else X[context['held_rows']]
        probability = predict_model(packet, raw)
    context['models'][name] = packet
    context['probabilities'][name] = probability
    directory = OUT/'models'/context['name']
    directory.mkdir(exist_ok=True)
    path = directory/f'{name}.joblib'
    joblib.dump(packet, path, compress=3)
    reloaded = joblib.load(path)
    if config['stage'] != 'blend':
        raw = XV if context['name'] == 'full_training' else X[context['held_rows']]
        np.testing.assert_allclose(predict_model(reloaded, raw), probability, rtol=1e-12, atol=1e-12)
    else:
        db_packet, gm_packet = [joblib.load(directory/f) for f in reloaded['component_files']]
        raw = XV if context['name'] == 'full_training' else X[context['held_rows']]
        replay = (1.-reloaded['gmm_weight'])*predict_model(db_packet, raw) + reloaded['gmm_weight']*predict_model(gm_packet, raw)
        np.testing.assert_allclose(replay, probability, rtol=1e-12, atol=1e-12)
    return probability


def persist_predictions(context, config):
    probability = fit_config(context, config)
    if context['name'] == 'full_training':
        frame = valid[['ID']].copy()
        truth = YV
    else:
        frame = train.iloc[context['held_rows']][['ID']].copy()
        truth = Y[context['held_rows']]
    prediction, empty = label_predictions(probability)
    for j, reason in enumerate(REASONS):
        frame[f'actual_{reason}'] = truth[:, j]
        frame[f'probability_{reason}'] = probability[:, j]
        frame[f'predicted_{reason}'] = prediction[:, j]
    frame['used_fallback'] = empty
    denominator = truth.sum(axis=1)+prediction.sum(axis=1)
    frame['trial_f1'] = np.divide(2*(truth*prediction).sum(axis=1), denominator,
                                 out=np.zeros(len(truth), dtype=float), where=denominator != 0)
    directory = OUT/'predictions'/context['name']
    directory.mkdir(exist_ok=True)
    frame.to_csv(directory/f"{config['name']}.csv", index=False)
    return dict(context=context['name'], repeat=context['repeat'], fold=context['fold'],
                configuration=config['name'], stage=config['stage'], held_rows=len(truth), **score(truth, probability))



## Exact fold assignments

Five repeats of three-fold multilabel stratification balance rare reasons as closely as possible. Each labeled trial is held out once per repeat.

In [3]:
# Generate folds before computing any candidate scores. The external validation set
# is never used to stratify, fit, calibrate, or generate pseudo-labels.
SPLITS = []
split_audit = []
for repeat, seed in enumerate(SPLIT_SEEDS):
    splitter = MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    coverage = np.zeros(len(known_rows), dtype=int)
    assignment = train.iloc[known_rows][['ID']].copy()
    assignment['repeat'] = repeat
    assignment['seed'] = seed
    assignment['fold'] = -1
    for fold, (fit_local, held_local) in enumerate(splitter.split(X[known], Y[known])):
        fit_rows = known_rows[fit_local]
        held_rows = known_rows[held_local]
        coverage[held_local] += 1
        assignment.iloc[held_local, assignment.columns.get_loc('fold')] = fold
        assert len(np.intersect1d(fit_rows, held_rows)) == 0
        for j, reason in enumerate(REASONS):
            assert 0 < Y[fit_rows, j].sum() < len(fit_rows)
            split_audit.append(dict(repeat=repeat, seed=seed, fold=fold, reason=reason,
                                    fit_rows=len(fit_rows), held_rows=len(held_rows),
                                    fit_positive=int(Y[fit_rows, j].sum()),
                                    held_positive=int(Y[held_rows, j].sum())))
        SPLITS.append(dict(name=f'repeat{repeat}_fold{fold}', repeat=repeat, fold=fold,
                           fitting_known_rows=fit_rows, held_rows=held_rows))
    np.testing.assert_array_equal(coverage, np.ones(len(known_rows), dtype=int))
    assignment.to_csv(OUT/'splits'/f'repeat{repeat}.csv', index=False)
pd.DataFrame(split_audit).to_csv(OUT/'splits/label_balance.csv', index=False)
print('Held-out positives per fold, across repeats:')
display(pd.DataFrame(split_audit).groupby('reason', sort=False).held_positive.agg(['min', 'max']))



Held-out positives per fold, across repeats:


,min,max
reason,,
Safety,5,6
Funding,2,3
Efficacy,5,6
Recruitment,33,34
Other,8,9


## Reproduce existing models

Verify the full-data implementation against the saved supervised, DBSCAN, and GMM models before running cross-validation.

In [4]:
# Reproduce every baseline on all 157 training labels before trusting the new
# fold implementation. No candidate selection uses these validation scores.
full = build_context('full_training', known_rows, np.array([], dtype=int))
full_rows = []
for config in CONFIGS[:3]:
    row = persist_predictions(full, config)
    full_rows.append(row)
    name = config['name']
    if name == 'supervised':
        original_probability = joblib.load(SUPERVISED_MODEL).predict_proba(XV)
    else:
        original_model = joblib.load(DB_MODEL if name == 'dbscan' else GMM_MODEL)
        original_probability = predict_model(original_model, XV)
        source = SOURCE/'datasets/dbscan_m5_q25.csv' if name == 'dbscan' else DATA/'experiments/method_comparison/datasets/gmm_tied_k10.csv'
        saved = pd.read_csv(source, float_precision='round_trip')
        assert saved.ID.tolist() == train.ID.tolist()
        np.testing.assert_array_equal(full['datasets'][name][0], saved[REASONS].to_numpy(dtype=int))
        np.testing.assert_allclose(full['datasets'][name][1], saved[[f'weight_{r}' for r in REASONS]].to_numpy(), rtol=1e-10, atol=1e-12)
    np.testing.assert_allclose(full['probabilities'][name], original_probability, rtol=1e-8, atol=1e-10)
print('Rebuilt all three full-training baselines and reproduced saved targets, weights, and validation probabilities.')
display(pd.DataFrame(full_rows)[['configuration', 'sample_f1', 'macro_f1']])



Rebuilt all three full-training baselines and reproduced saved targets, weights, and validation probabilities.


,configuration,sample_f1,macro_f1
0,supervised,0.893162,0.804324
1,dbscan,0.910256,0.811421
2,gmm,0.905983,0.811626


## Compare the three baseline methods

Rebuild preprocessing, clustering, label generation, and classifiers independently in every fold.

In [5]:
CONTEXTS = []
fold_rows = []
for spec in SPLITS:
    context = build_context(**spec)
    CONTEXTS.append(context)
    for config in CONFIGS[:3]:
        fold_rows.append(persist_predictions(context, config))
    pd.DataFrame(fold_rows).to_csv(OUT/'fold_scores.csv', index=False)
    pd.DataFrame(CONTEXT_AUDIT).to_csv(OUT/'context_audit.csv', index=False)
    pd.DataFrame(LABEL_AUDIT).to_csv(OUT/'pseudo_label_audit.csv', index=False)
    latest = {r['configuration']: round(r['sample_f1'], 5) for r in fold_rows[-3:]}
    print(context['name'], latest, flush=True)
print('Baseline cross-validation complete. Continuing the three predeclared classifier experiments on these same contexts.')



repeat0_fold0 {'supervised': 0.92767, 'dbscan': 0.92767, 'gmm': 0.95283}


repeat0_fold1 {'supervised': 0.89542, 'dbscan': 0.89542, 'gmm': 0.91503}


repeat0_fold2 {'supervised': 0.86478, 'dbscan': 0.86478, 'gmm': 0.86164}


repeat1_fold0 {'supervised': 0.83013, 'dbscan': 0.83013, 'gmm': 0.83013}


repeat1_fold1 {'supervised': 0.89744, 'dbscan': 0.89744, 'gmm': 0.85897}


repeat1_fold2 {'supervised': 0.90566, 'dbscan': 0.90566, 'gmm': 0.89937}


... (6 outputs left out)

repeat4_fold0 {'supervised': 0.87179, 'dbscan': 0.87179, 'gmm': 0.88141}


repeat4_fold1 {'supervised': 0.91824, 'dbscan': 0.91824, 'gmm': 0.88679}


repeat4_fold2 {'supervised': 0.86218, 'dbscan': 0.86218, 'gmm': 0.875}


Baseline cross-validation complete. Continuing the three predeclared classifier experiments on these same contexts.


## Tolerance, normalized embeddings, and probability blends

Evaluate the three proposed changes on the same fold populations. Do not combine changes or search new grids after observing scores.

In [6]:
# Each change is isolated: tolerance only; normalization + a C grid only;
# probability blends of the unchanged original DBSCAN and GMM classifiers only.
for context in CONTEXTS:
    for config in CONFIGS[3:]:
        fold_rows.append(persist_predictions(context, config))
    pd.DataFrame(fold_rows).to_csv(OUT/'fold_scores.csv', index=False)
    pd.DataFrame(FIT_AUDIT).to_csv(OUT/'fit_audit.csv', index=False)
    print('Completed all 16 configurations:', context['name'], flush=True)
assert len(fold_rows) == 15*16



Completed all 16 configurations: repeat0_fold0


Completed all 16 configurations: repeat0_fold1


Completed all 16 configurations: repeat0_fold2


Completed all 16 configurations: repeat1_fold0


Completed all 16 configurations: repeat1_fold1


Completed all 16 configurations: repeat1_fold2


... (5 outputs left out)

Completed all 16 configurations: repeat3_fold2


Completed all 16 configurations: repeat4_fold0


Completed all 16 configurations: repeat4_fold1


Completed all 16 configurations: repeat4_fold2


## Pool cross-validation predictions and freeze choices

Pool all held-out predictions in each repeat, then report mean sample F1 and descriptive variation across repeats. Select configurations using CV before assessing their external validation scores.

In [7]:
# Aggregate all 157 out-of-fold trials within each repeat. Sample F1 is pooled
# by trial, avoiding unequal-fold-size averaging. Repeats reuse the same trials.
repeat_rows = []
for repeat in range(len(SPLIT_SEEDS)):
    contexts = [c for c in CONTEXTS if c['repeat'] == repeat]
    rows = np.concatenate([c['held_rows'] for c in contexts])
    assert sorted(rows.tolist()) == known_rows.tolist()
    for config in CONFIGS:
        probability = np.vstack([c['probabilities'][config['name']] for c in contexts])
        repeat_rows.append(dict(repeat=repeat, seed=SPLIT_SEEDS[repeat], configuration=config['name'],
                                stage=config['stage'], **score(Y[rows], probability)))
repeats = pd.DataFrame(repeat_rows)
folds = pd.DataFrame(fold_rows)
base_fold = folds[folds.configuration == 'dbscan'].set_index('context').sample_f1
folds['delta_vs_dbscan'] = folds.sample_f1 - folds.context.map(base_fold)
base_repeat = repeats[repeats.configuration == 'dbscan'].set_index('repeat').sample_f1
repeats['delta_vs_dbscan'] = repeats.sample_f1 - repeats['repeat'].map(base_repeat)
repeats.to_csv(OUT/'repeat_scores.csv', index=False)
folds.to_csv(OUT/'fold_scores.csv', index=False)
summary_rows = []
for index, config in enumerate(CONFIGS):
    r = repeats[repeats.configuration == config['name']]
    f = folds[folds.configuration == config['name']]
    summary_rows.append(dict(configuration=config['name'], stage=config['stage'], order=index,
                             cv_sample_f1=float(r.sample_f1.mean()), cv_repeat_sd=float(r.sample_f1.std(ddof=1)),
                             cv_macro_f1=float(r.macro_f1.mean()),
                             cv_min_repeat=float(r.sample_f1.min()), cv_max_repeat=float(r.sample_f1.max()),
                             cv_delta_vs_dbscan=float(r.delta_vs_dbscan.mean()),
                             fold_wins=int((f.delta_vs_dbscan > 1e-12).sum()),
                             fold_ties=int((f.delta_vs_dbscan.abs() <= 1e-12).sum()),
                             fold_losses=int((f.delta_vs_dbscan < -1e-12).sum())))
cv_summary = pd.DataFrame(summary_rows)
cv_summary.to_csv(OUT/'cv_comparison.csv', index=False)
cv_ranked = cv_summary.sort_values(['cv_sample_f1', 'order'], ascending=[False, True], kind='stable')
display(cv_ranked[['configuration', 'cv_sample_f1', 'cv_repeat_sd', 'cv_delta_vs_dbscan', 'fold_wins', 'fold_ties', 'fold_losses']])
# Freeze choices from CV before assessing external validation for new settings.
selected = {stage: cv_ranked[cv_ranked.stage == stage].iloc[0].configuration
            for stage in ['tolerance', 'normalization', 'blend']}
selected['overall'] = cv_ranked.iloc[0].configuration
(OUT/'cv_selection.json').write_text(json.dumps(dict(
    criterion='Mean pooled out-of-fold sample F1 over five repeats; ties use predeclared order',
    choices=selected, external_validation_used_for_selection=False), indent=2)+'\n')
print('CV-selected configurations:', selected)



,configuration,cv_sample_f1,cv_repeat_sd,cv_delta_vs_dbscan,fold_wins,fold_ties,fold_losses
9,normalized_C100,0.909130,0.008676,0.017240,14,0,1
11,normalized_C10000,0.907856,0.006117,0.015966,12,1,2
12,normalized_C100000,0.907856,0.006117,0.015966,12,1,2
8,normalized_C10,0.907856,0.010026,0.015966,14,0,1
10,normalized_C1000,0.907856,0.009327,0.015966,12,0,3
3,tolerance_0p001,0.894013,0.009146,0.002123,4,8,3
14,blend_gmm_0p5,0.894013,0.019044,0.002123,5,5,5
0,supervised,0.891890,0.011105,0.000000,0,15,0
1,dbscan,0.891890,0.011105,0.000000,0,15,0
6,tolerance_1em05,0.891720,0.007042,-0.000170,7,2,6


CV-selected configurations: {'tolerance': 'tolerance_0p001', 'normalization': 'normalized_C100', 'blend': 'blend_gmm_0p5', 'overall': 'normalized_C100'}


## Compare with the existing 78-trial validation set

Train all candidates with the original 157 verified labels and score the separate validation set. The current saved model remains unchanged.

In [8]:
for config in CONFIGS[3:]:
    full_rows.append(persist_predictions(full, config))
validation = pd.DataFrame(full_rows)
baseline_validation = validation[validation.configuration == 'dbscan'].iloc[0]
validation['validation_delta_vs_dbscan'] = validation.sample_f1 - baseline_validation.sample_f1
baseline_prediction = label_predictions(full['probabilities']['dbscan'])[0]
validation['changed_validation_trials'] = [int((label_predictions(full['probabilities'][name])[0] != baseline_prediction).any(axis=1).sum())
                                         for name in validation.configuration]
validation.to_csv(OUT/'validation_scores.csv', index=False)
comparison = cv_summary.merge(validation[['configuration', 'sample_f1', 'macro_f1',
                                           'validation_delta_vs_dbscan', 'changed_validation_trials']], on='configuration')
comparison = comparison.rename(columns={'sample_f1': 'validation_sample_f1', 'macro_f1': 'validation_macro_f1'})
comparison['selected_by_cv_in_stage'] = [name in selected.values() for name in comparison.configuration]
comparison.to_csv(OUT/'comparison.csv', index=False)
pd.DataFrame(CONTEXT_AUDIT).to_csv(OUT/'context_audit.csv', index=False)
pd.DataFrame(LABEL_AUDIT).to_csv(OUT/'pseudo_label_audit.csv', index=False)
pd.DataFrame(FIT_AUDIT).to_csv(OUT/'fit_audit.csv', index=False)
display(comparison[['configuration', 'cv_sample_f1', 'cv_delta_vs_dbscan',
                    'validation_sample_f1', 'validation_delta_vs_dbscan', 'changed_validation_trials']])



,configuration,cv_sample_f1,cv_delta_vs_dbscan,validation_sample_f1,validation_delta_vs_dbscan,changed_validation_trials
0,supervised,0.891890,0.000000,0.893162,-0.017094,3
1,dbscan,0.891890,0.000000,0.910256,0.000000,0
2,gmm,0.889130,-0.002760,0.905983,-0.004274,2
3,tolerance_0p001,0.894013,0.002123,0.897436,-0.012821,1
4,tolerance_0p0003,0.891677,-0.000212,0.897436,-0.012821,1
5,tolerance_3em05,0.891465,-0.000425,0.897436,-0.012821,2
6,tolerance_1em05,0.891720,-0.000170,0.884615,-0.025641,6
7,normalized_C1,0.847558,-0.044331,0.794872,-0.115385,18
8,normalized_C10,0.907856,0.015966,0.829060,-0.081197,12
9,normalized_C100,0.909130,0.017240,0.829060,-0.081197,10


## Verify saved models and independent refits

Reload every saved model, replay selected fitting contexts using exported weights, and verify protected input hashes.

In [9]:
# Independent replay of a fold and all full-training non-blend configurations
# from saved targets and normalized weights. Fitting never reads held-out labels.
replay_checks = []
for context in [CONTEXTS[0], full]:
    raw = XV if context is full else X[context['held_rows']]
    dataset_cache = {}
    for config in CONFIGS:
        if config['stage'] == 'blend':
            continue
        name = config['name']
        saved_model = joblib.load(OUT/'models'/context['name']/f'{name}.joblib')
        dataset_name = config['dataset']
        if dataset_name not in dataset_cache:
            dataset_cache[dataset_name] = pd.read_csv(OUT/'contexts'/context['name']/f'{dataset_name}_targets_weights.csv', float_precision='round_trip')
        frame = dataset_cache[dataset_name]
        pool_rows = pd.Index(train.ID).get_indexer(frame.ID)
        assert (pool_rows >= 0).all() and not len(np.intersect1d(pool_rows, context['held_rows']))
        design = saved_model['scaler'].transform(X[pool_rows])
        if saved_model['normalizer'] is not None:
            design = saved_model['normalizer'].transform(design)
        replay = {**saved_model, 'estimators': []}
        with threadpool_limits(limits=1):
            for j, reason in enumerate(REASONS):
                target = frame[reason].to_numpy(dtype=int)
                weight = frame[f'normalized_weight_{reason}'].to_numpy()
                included = weight > 0
                estimator = clone(saved_model['estimators'][j])
                with warnings.catch_warnings():
                    warnings.simplefilter('error', ConvergenceWarning)
                    estimator.fit(design[included], target[included], sample_weight=weight[included])
                replay['estimators'].append(estimator)
        np.testing.assert_allclose(predict_model(replay, raw), context['probabilities'][name], rtol=1e-8, atol=1e-10)
        replay_checks.append(dict(context=context['name'], configuration=name, passed=True))
pd.DataFrame(replay_checks).to_csv(OUT/'replay_checks.csv', index=False)
for relative, digest in PROTECTED_HASHES.items():
    assert hashlib.sha256((DATA/relative).read_bytes()).hexdigest() == digest
assert len(CONTEXT_AUDIT) == 16 and len(full_rows) == 16 and len(repeats) == 80
assert all(row['heldout_fit_overlap'] == 0 and row['gmm_converged'] for row in CONTEXT_AUDIT)
print('Verified: 240 fold evaluations, 80 pooled repeat evaluations, 16 external validation evaluations;')
print('all saved models reload; 26 independent refits reproduce probabilities; protected input hashes unchanged.')



Verified: 240 fold evaluations, 80 pooled repeat evaluations, 16 external validation evaluations;
all saved models reload; 26 independent refits reproduce probabilities; protected input hashes unchanged.


## Save the experiment record

Record all grids, scores, exact splits, fitting populations, audit data, selections, dependencies, and limitations.

In [10]:
settings = dict(
    split_seeds=SPLIT_SEEDS, folds=N_FOLDS, repeats=len(SPLIT_SEEDS),
    stratifier='iterstrat.MultilabelStratifiedKFold', reason_order=REASONS, features=FEATURES,
    cv_verified_rows=157, external_validation_rows=78, unlabeled_training_rows=1014,
    heldout_trials_removed_from_all_fitting=True, heldout_features_used_for_fit=False,
    label_rule=LABEL_RULE, gmm_rule=GMM_RULE,
    dbscan_rule=dict(min_samples=5, eps='training-pool 25th percentile of fifth-neighbor distances, floor 1e-6'),
    classifier_preprocessing='StandardScaler fitted only to the verified fitting rows in each context',
    cluster_preprocessing='Separate StandardScaler fitted to remaining verified + 1014 unlabeled training rows',
    fitting_weight_normalization='sum equals verified fitting-row count for each reason',
    normalization='StandardScaler then row-wise L2 Normalizer; pseudo-labels fixed within each fold',
    classifier_solver='lbfgs', classifier_max_iter=5000, classifier_random_state=42,
    configurations=CONFIGS, cutoff=0.5, highest_probability_fallback=True,
    paired_comparison='same trials/splits across all candidates; pool trials within repeat then average repeats',
    dispersion='repeat SD is descriptive; repeats share trials and are not independent test samples',
    temporal_caveat='Random folds within early/rare-enriched training labels do not reproduce the later-trial Kaggle distribution',
    selection=selected, selection_external_validation_used=False,
    existing_validation_repeatedly_consulted=True, leader_changed=False,
    current_model_retained=str(DB_MODEL.relative_to(DATA)),
    final_validation_label_refit=False, new_kaggle_scores=None,
    binary_fits=len(FIT_AUDIT), protected_input_sha256=PROTECTED_HASHES,
    dataset_file_sha256={str(p.relative_to(OUT)): hashlib.sha256(p.read_bytes()).hexdigest()
                         for p in sorted((OUT/'datasets').glob('*.csv'))},
    versions=dict(python=platform.python_version(), sklearn=sklearn.__version__, numpy=np.__version__,
                  pandas=pd.__version__, iterative_stratification=importlib.metadata.version('iterative-stratification')))
(OUT/'settings.json').write_text(json.dumps(settings, indent=2)+'\n')

lines = ['# Repeated cross-validation and three classifier experiments', '',
         'Five predeclared repeats of three-fold multilabel-stratified CV use the 157 verified training trials. '
         'Each fold fits about 105 verified trials plus the 1,014 originally unlabeled trials for clustering. '
         'Held-out trials are excluded entirely from classifier scaling, cluster scaling, clustering, label mapping, '
         'pseudo-label generation, and classifier fitting. All methods share the same splits. '
         'The external 78-trial validation set is never used for fitting or CV selection.', '',
         'Three full-training baseline recipes reproduce their saved targets, raw weights, and validation probabilities. '
         'Pseudo-labels are rebuilt per fold, then frozen while testing classifier changes. '
         'Weights total the number of verified fitting trials per reason. '
         'Classifier/cluster scalers are refit within each fold; DBSCAN keeps its training-derived q25 radius rule '
         'rather than a radius taken from all training trials.', '',
         '## Results', '',
         '| Configuration | CV sample F1 | Repeat SD | CV change vs DBSCAN | Fold wins/ties/losses | Validation sample F1 | Validation change |',
         '|---|---:|---:|---:|---|---:|---:|']
for row in comparison.itertuples():
    lines.append(f'| {row.configuration} | {row.cv_sample_f1:.6f} | {row.cv_repeat_sd:.6f} | '
                 f'{row.cv_delta_vs_dbscan:+.6f} | {row.fold_wins}/{row.fold_ties}/{row.fold_losses} | '
                 f'{row.validation_sample_f1:.6f} | {row.validation_delta_vs_dbscan:+.6f} |')
lines += ['', 'CV sample F1 pools all 157 out-of-fold trials within each repeat and averages the five repeat scores. '
          'Repeat SD and paired fold win/tie/loss counts describe stability; repeats and folds overlap in training '
          'and do not supply independent-test confidence intervals.', '',
          '## Predeclared experiments', '',
          '- Baselines: supervised logistic, DBSCAN + logistic, tied-covariance GMM k=10 + logistic; C=1000, tol=1e-4.',
          '- Tolerance: 1e-3, 3e-4, 1e-4, 3e-5, 1e-5; fixed DBSCAN data and C=1000.',
          '- Embedding normalization: StandardScaler then unit L2 length per trial; fixed DBSCAN targets, '
          'C=1/10/100/1000/10000/100000 and tol=1e-4. These C values account for the feature-scale change.',
          '- Blend: probability average of the original DBSCAN and GMM classifiers; GMM weights 0/0.25/0.5/0.75/1, '
          'with endpoints reused from baseline models. Threshold/fallback remain fixed.', '',
          'CV selections were recorded before evaluating the new settings on the external validation set: '
          + json.dumps(selected, sort_keys=True) + '.', '',
          '## Interpretation limits and retained model', '',
          'The 157 training trials are early/rare-enriched, whereas validation/Kaggle trials are later. Random CV '
          'therefore measures split stability within the training distribution, not the time shift. It also uses '
          'fewer verified fitting trials per model. CV scores and the external validation score are different '
          'evaluation settings and should not be compared as if they were the same metric estimate. '
          'Both have influenced exploratory model development; neither is a fresh independent final test.', '',
          'Keep the saved current DBSCAN model and prepared Kaggle file unchanged pending review. '
          'No final train+validation refit, test prediction, Kaggle submission, commit, push, or tag was performed.', '',
          '## Reproduction and saved artifacts', '',
          'Notebook: part1/classifier_cross_validation.ipynb. Additional dependency: iterative-stratification==0.1.9. '
          'splits/ stores all exact fold assignments and per-label balances. contexts/ stores row roles, '
          'pseudo-label targets/raw and normalized weights, scalers, cluster models, responsibilities and support. '
          'models/ and predictions/ store every context/configuration. Baselines are fully rebuilt inside folds, '
          'not borrowed from full-training generated labels. fit_audit.csv records actual binary fits, '
          'context_audit.csv records fitting populations/convergence, and pseudo_label_audit.csv records generated labels. '
          'All 256 saved candidate packets reload with matching probabilities; 26 independent classifier refits '
          'from persisted targets/normalized weights reproduce probabilities. settings.json records grids, '
          'selection, versions, checksums, and safeguards.']
(OUT/'SUMMARY.md').write_text('\n'.join(lines)+'\n')
print('Saved:', OUT/'SUMMARY.md')


Saved: /Users/michael/Documents/School/Cornell/AppliedMachineLearning/AML-Midterm/part1/experiments/classifier_cross_validation/SUMMARY.md


## Findings and independent disk checks

Rescore every saved prediction file, verify fold isolation, and document where cross-validation and supplied validation disagree. No candidate is promoted automatically.

In [11]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

cwd = Path.cwd()
part1 = next(p for p in [cwd, cwd/'part1', cwd/'AML-Midterm'/'part1']
             if (p/'experiments/classifier_cross_validation/settings.json').is_file()).resolve()
experiment = part1/'experiments/classifier_cross_validation'
settings = json.loads((experiment/'settings.json').read_text())
comparison = pd.read_csv(experiment/'comparison.csv')
fold_scores = pd.read_csv(experiment/'fold_scores.csv')
validation_scores = pd.read_csv(experiment/'validation_scores.csv')
original = pd.read_csv(experiment/'datasets/original_training.csv')
reasons = settings['reason_order']
audit = pd.read_csv(experiment/'pseudo_label_audit.csv')
audit['pseudo_slots'] = audit.pseudo_positive + audit.pseudo_negative
fold_generation = audit[(audit.dataset == 'dbscan') & (audit.context != 'full_training')].groupby('context').pseudo_slots.sum()
zero_generation = int((fold_generation == 0).sum())
nonzero_generation = int((fold_generation > 0).sum())
assert zero_generation == 14 and nonzero_generation == 1

score_checks = []
for row in pd.concat([fold_scores, validation_scores], ignore_index=True).itertuples():
    frame = pd.read_csv(experiment/'predictions'/row.context/f'{row.configuration}.csv', float_precision='round_trip')
    actual = frame[[f'actual_{r}' for r in reasons]].to_numpy(dtype=int)
    probability = frame[[f'probability_{r}' for r in reasons]].to_numpy()
    prediction = (probability >= 0.5).astype(int)
    empty = prediction.sum(axis=1) == 0
    indices = np.flatnonzero(empty)
    prediction[indices, probability[indices].argmax(axis=1)] = 1
    np.testing.assert_array_equal(prediction, frame[[f'predicted_{r}' for r in reasons]].to_numpy(dtype=int))
    sample = f1_score(actual, prediction, average='samples', zero_division=0)
    macro = f1_score(actual, prediction, average='macro', zero_division=0)
    np.testing.assert_allclose([sample, macro], [row.sample_f1, row.macro_f1], atol=1e-14)
    score_checks.append(dict(context=row.context, configuration=row.configuration, passed=True))
    if row.context != 'full_training':
        roles = pd.read_csv(experiment/'contexts'/row.context/'row_roles.csv')
        assert set(frame.ID) == set(roles.loc[roles.role == 'held_out', 'ID'])
        for dataset in ['supervised', 'dbscan', 'gmm']:
            fitting = pd.read_csv(experiment/'contexts'/row.context/f'{dataset}_targets_weights.csv')
            assert set(frame.ID).isdisjoint(fitting.ID)
        if row.configuration == 'dbscan':
            supervised = pd.read_csv(experiment/'predictions'/row.context/'supervised.csv')
            np.testing.assert_array_equal(frame[[f'predicted_{r}' for r in reasons]],
                                          supervised[[f'predicted_{r}' for r in reasons]])
pd.DataFrame(score_checks).to_csv(experiment/'disk_score_checks.csv', index=False)
assert len(score_checks) == 256
for section, root in [('protected_input_sha256', part1), ('dataset_file_sha256', experiment)]:
    for relative, digest in settings[section].items():
        assert hashlib.sha256((root/relative).read_bytes()).hexdigest() == digest

settings['temporal_caveat'] = (
    'Random CV resamples the 157 early/rare-enriched labels and fits 103-106 verified rows per fold. '
    'It does not reproduce the supplied validation population or necessarily the Kaggle distribution; '
    'CV/validation disagreement alone does not identify its cause.')
settings['verified_fit_rows_per_fold'] = dict(min=103, max=106)
settings['held_rows_per_fold'] = dict(min=51, max=54)
settings['dbscan_folds_without_generated_labels'] = zero_generation
settings['dbscan_folds_with_generated_labels'] = nonzero_generation
settings['independently_rescored_prediction_files'] = len(score_checks)
(experiment/'settings.json').write_text(json.dumps(settings, indent=2)+'\n')

findings = [
    '## Findings', '',
    '- No tested change exceeds the current 0.910256 sample F1 on the supplied 78 validation trials. '
    'A 25% GMM blend ties it with identical predicted labels.',
    '- Supervised and DBSCAN predictions match on all 15 CV folds, averaging 0.891890 sample F1. '
    'DBSCAN adds no pseudo-labels in 14 folds; the remaining fold adds 81 reason labels without changing '
    'held-out predicted labels. This is limited evidence about its full-157-label benefit: its '
    'generation rule usually abstains with only 103-106 verified fitting trials.',
    '- GMM averages 0.889130, slightly below the supervised/DBSCAN CV controls; it wins 6 folds, '
    'ties 2, and loses 7 against DBSCAN.',
    '- The CV-selected tolerance 1e-3 averages 0.894013, a 0.002123 gain, with 4 fold wins, '
    '8 ties, and 3 losses. It scores 0.897436 on supplied validation; retain original tol=1e-4.',
    '- Unit-length embeddings at C=100 average 0.909130 in CV, a 0.017240 gain, with 14 fold wins '
    'and 1 loss. They score 0.829060 on supplied validation. The best validation score among '
    'normalized variants is 0.863248 at C=10000/100000, also below the current model.',
    '- The CV-selected 50% GMM blend averages 0.894013, with 5 fold wins, 5 ties, and 5 losses. '
    'It scores 0.901709 on supplied validation. Blending has not established a consistent improvement.', '',
    'Normalization is a real CV improvement in this run, but it does not transfer to the supplied '
    'validation rows. Differences in fitting-set size and sampled populations can affect rankings. '
    'The data here do not isolate the cause. Random folds draw from the same 157 early/rare-enriched '
    'training labels; they do not recreate the supplied validation population or guarantee Kaggle rankings. '
    'Keep the current saved model while retaining the alternative fits for review.', '',
    'Verification independently rescored all 256 saved prediction files and checked fitting/held-out ID '
    'separation. All protected source-model, source-data, and current Kaggle-file checksums are unchanged.'
]
summary_path = experiment/'SUMMARY.md'
summary = summary_path.read_text()
old = ('The 157 training trials are early/rare-enriched, whereas validation/Kaggle trials are later. Random CV '
       'therefore measures split stability within the training distribution, not the time shift. It also uses '
       'fewer verified fitting trials per model.')
new = ('The 157 training labels are early/rare-enriched. Random CV measures split stability within those '
       'rows, fitting 103-106 verified trials per fold instead of 157. It does not reproduce the supplied '
       'validation population or necessarily the Kaggle distribution; disagreement alone does not establish its cause.')
summary = summary.replace(old, new)
summary = summary.split('\n## Findings\n')[0].rstrip()+'\n\n'+'\n'.join(findings)+'\n'
summary_path.write_text(summary)
print('Independent disk verification passed for 256 prediction files, fold isolation, and all protected hashes.')
print('Saved interpretation, including DBSCAN abstention and CV/validation ranking disagreement.')


Independent disk verification passed for 256 prediction files, fold isolation, and all protected hashes.
Saved interpretation, including DBSCAN abstention and CV/validation ranking disagreement.
